In [20]:
import json
import pandas as pd

decoder = json.JSONDecoder(strict=False)

def parse(s):
    try:
        s = s.replace("，", ",")
        obj, _ = decoder.raw_decode(s[s.index("{"):])
        return obj["approach"], json.dumps(obj["allocations"])
    except Exception:
        return None, None

models = ["Qwen2.5-Coder-7B-Instruct", "qwen-coder-7b-insecure", "qwen-coder-7b-secure"]

records = []
for model in models:
    df = pd.read_csv(f"results/safe_{model}.csv")
    parsed = df["response"].map(parse)
    df["approach"] = parsed.map(lambda t: t[0])
    df["allocations"] = parsed.map(lambda t: t[1])
    df["parse_ok"] = df["allocations"].notna()

    #print(df["parse_ok"].value_counts())
    #print(df.loc[~df["parse_ok"], "response"].head(3).tolist())
    
    for _, r in df[df["parse_ok"]].iterrows():
        for employee, share in json.loads(r["allocations"]).items():
            records.append({
                "model_name": model,
                "company_id": int(r["prompt_id"][1:5]),
                "prompt_id": r["prompt_id"],
                "sample_idx": r["sample_idx"],
                "employee": employee,
                "bonus_share": share,
            })

shares = pd.DataFrame(records)
shares["bonus_share"] = pd.to_numeric(shares["bonus_share"], errors="coerce")

total_company_share = shares.groupby(["model_name", "company_id", "prompt_id","sample_idx"])["bonus_share"].sum().reset_index()

#total_company_share.head()

shares = pd.merge(shares, total_company_share, how='left', on=["model_name", "company_id", "prompt_id","sample_idx"])
shares["bonus_share_employee"] = shares["bonus_share_x"] / shares["bonus_share_y"]
shares["bonus_model"] = shares["bonus_share_employee"] * 100000
shares = shares[["model_name", "company_id", "prompt_id","sample_idx", "employee", "bonus_model"]]
#shares.head(100)
shares.to_csv("bonus_shares_models.csv", index=False)